In [12]:
!pip  install trl==0.20.0
!pip install -U bitsandbytes GPUtil psutil

  Attempting uninstall: trl
    Found existing installation: trl 0.23.0
    Uninstalling trl-0.23.0:
      Successfully uninstalled trl-0.23.0


In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import os
from transformers import BitsAndBytesConfig
import subprocess as sp
import os
from trl import SFTConfig, SFTTrainer, setup_chat_format

def get_gpu_memory():
    command = "nvidia-smi --query-gpu=memory.free --format=csv"
    memory_free_info = sp.check_output(command.split()).decode('ascii').split('\n')[:-1][1:]
    memory_free_values = [int(x.split()[0]) for i, x in enumerate(memory_free_info)]
    return memory_free_values

get_gpu_memory()

2025-09-18 17:23:30.934657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758216210.950360   26073 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758216210.955302   26073 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-18 17:23:30.972028: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[22503]

In [2]:
model_name = "HuggingFaceTB/SmolLM2-360M"
#dataset_name = "HuggingFaceTB/smoltalk"
model_cache_dir=model_name.split('/')[-1]
dataset_name = 'openai/gsm8k'
config_name = 'main'
#config_name = "smol-summarize"
dataset_cache_dir=f"{dataset_name.replace('/', '')}_{config_name}"
output_dir = "./test_gmsk8_smol"

In [3]:
#Load the model and tokenizer

model = AutoModelForCausalLM.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir,
    device_map='cuda',
)
tokenizer = AutoTokenizer.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir
)

In [4]:
get_gpu_memory()

[20863]

In [5]:
# load dataset

ds = load_dataset(dataset_name, config_name, cache_dir=dataset_cache_dir)
ds

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [6]:
os.environ["WANDB_PROJECT"] = "llmops-project"  # name your W&B project
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # log all model checkpoints
# https://docs.wandb.ai/guides/integrations/huggingface/

In [8]:
# Configure trainer

training_args = SFTConfig(
    output_dir=output_dir,
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=100,
    save_steps=400,
    eval_strategy="steps",
    eval_steps=200,
    report_to="wandb",
    run_name="trl"
)

# Initialize trainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,
)

In [ ]:
# Start training
trainer.train()

# Save the model
trainer.save_model(output_dir)

wandb: Currently logged in as: creigner (creigner-axa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
200,1.688800,1.778626


In [13]:
torch.cuda.empty_cache()

In [14]:
del model

NameError: name 'model' is not defined

In [7]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import wandb
import time
import psutil
import GPUtil
from datetime import datetime

# Configuration
max_steps = 5000
batch_size = 8
learning_rate = 5e-5
logging_steps = 100
save_steps = 400
eval_steps = 200
n_epochs = 10
max_length = 512
run_name = f'llmops-{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}'
# Initialize wandb with more config
wandb.init(
    project="llmops-test",
    name=run_name,
    config={
        "max_steps": max_steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_epochs": n_epochs,
        "max_length": max_length
    }
)

# Prepare data
train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(ds["test"], batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()
tokenizer.pad_token = tokenizer.eos_token
model.train()

# Timing and monitoring setup
start_time = time.time()
step = 0
total_batches = len(train_loader)

print(f"Starting training: {n_epochs} epochs, {total_batches} batches per epoch")
print(f"Total steps planned: {min(max_steps, n_epochs * total_batches)}")

for epoch in range(n_epochs):
    if step >= max_steps:
        break
        
    epoch_start_time = time.time()
    epoch_loss = 0
    epoch_batches = 0
    
    print(f"\n=== Epoch {epoch + 1}/{n_epochs} ===")
    
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            break
            
        batch_start_time = time.time()
        
        inputs = tokenizer(
            batch["question"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )
        labels = tokenizer(
            batch["answer"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )["input_ids"]
        
        # Replace padding tokens in labels with -100
        labels[labels == tokenizer.pad_token_id] = -100
        
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_time = time.time() - batch_start_time
        epoch_loss += loss.item()
        epoch_batches += 1
        
        # Comprehensive logging
        if step % logging_steps == 0:
            elapsed_time = time.time() - start_time
            
            # GPU monitoring
            gpu_stats = {}
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats = {
                        "gpu_utilization": gpu.load * 100,
                        "gpu_memory_used": gpu.memoryUsed,
                        "gpu_memory_total": gpu.memoryTotal,
                        "gpu_memory_percent": (gpu.memoryUsed / gpu.memoryTotal) * 100,
                        "gpu_temperature": gpu.temperature
                    }
            except:
                pass
            
            # System stats
            cpu_percent = psutil.cpu_percent()
            memory = psutil.virtual_memory()
            
            # Training progress
            progress = (step / max_steps) * 100
            estimated_total_time = (elapsed_time / max(step, 1)) * max_steps
            eta = estimated_total_time - elapsed_time
            
            log_dict = {
                "train_loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]['lr'],
                "elapsed_time_minutes": elapsed_time / 60,
                "cpu_percent": cpu_percent,
                "memory_percent": memory.percent,
                "memory_used_gb": memory.used / (1024**3),
                **gpu_stats
            }
            
            wandb.log(log_dict)
            
            print(f"Step {step}/{max_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Batch time: {batch_time:.2f}s | "
                  f"ETA: {eta/60:.1f}min | "
                  f"GPU: {gpu_stats.get('gpu_memory_percent', 0):.1f}%")

        # Evaluation
        if step % eval_steps == 0 and step > 0:
            print(f"Running evaluation at step {step}...")
            eval_start_time = time.time()
            
            model.eval()
            eval_loss = 0
            eval_batches = 0
            with torch.no_grad():
                for eval_batch in eval_loader:
                    eval_inputs = tokenizer(
                        eval_batch["question"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )
                    eval_labels = tokenizer(
                        eval_batch["answer"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )["input_ids"]
                    
                    eval_labels[eval_labels == tokenizer.pad_token_id] = -100
                    
                    eval_inputs = {k: v.to(model.device) for k, v in eval_inputs.items()}
                    eval_labels = eval_labels.to(model.device)
                    
                    with autocast(device_type='cuda'):
                        eval_outputs = model(**eval_inputs, labels=eval_labels)
                        eval_loss += eval_outputs.loss.item()
                        eval_batches += 1
            
            avg_eval_loss = eval_loss / eval_batches
            eval_time = time.time() - eval_start_time
            
            wandb.log({
                "eval_loss": avg_eval_loss,
                "eval_time": eval_time,
                "step": step
            })
            
            print(f"Evaluation complete: Loss {avg_eval_loss:.4f} (took {eval_time:.2f}s)")
            model.train()

        # Save checkpoint
        if step % save_steps == 0 and step > 0:
            checkpoint_path = f"{output_dir}/model_step_{step}.pt"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint: {checkpoint_path}")
            wandb.log({"checkpoint_saved": step})

        step += 1

    # End of epoch logging
    epoch_time = time.time() - epoch_start_time
    avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
    
    wandb.log({
        "epoch_loss": avg_epoch_loss,
        "epoch_time": epoch_time,
        "epoch": epoch + 1,
        "batches_per_epoch": epoch_batches
    })
    
    print(f"Epoch {epoch + 1} complete: Avg loss {avg_epoch_loss:.4f}, Time: {epoch_time:.2f}s")

# Training complete
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
wandb.log({"total_training_time_minutes": total_time/60})

wandb.finish()

Starting training: 10 epochs, 935 batches per epoch
Total steps planned: 5000

=== Epoch 1/10 ===
Step 0/5000 (0.0%) | Loss: 4.1184 | Batch time: 0.44s | ETA: 42.0min | GPU: 68.5%
Step 100/5000 (2.0%) | Loss: 4.5468 | Batch time: 0.43s | ETA: 40.2min | GPU: 88.0%
Step 200/5000 (4.0%) | Loss: 4.2446 | Batch time: 0.43s | ETA: 39.2min | GPU: 88.0%
Running evaluation at step 200...
Evaluation complete: Loss 4.2867 (took 22.94s)
Step 300/5000 (6.0%) | Loss: 4.2298 | Batch time: 0.43s | ETA: 44.3min | GPU: 88.0%
Step 400/5000 (8.0%) | Loss: 4.3932 | Batch time: 0.43s | ETA: 41.9min | GPU: 88.0%
Running evaluation at step 400...
Evaluation complete: Loss 4.2543 (took 22.95s)
Saved checkpoint: ./test_gmsk8_smol/model_step_400.pt
Step 500/5000 (10.0%) | Loss: 4.0999 | Batch time: 0.43s | ETA: 46.2min | GPU: 88.0%
Step 600/5000 (12.0%) | Loss: 4.4035 | Batch time: 0.43s | ETA: 43.6min | GPU: 88.0%
Running evaluation at step 600...
Evaluation complete: Loss 4.2420 (took 22.95s)
Step 700/5000 (14

batches_per_epoch,█████▁
checkpoint_saved,▁▂▂▃▄▄▅▅▆▇▇█
cpu_percent,▁▄▄▄▄▄▄▄▄▄▄▄▄▇██▇██▆▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
elapsed_time_minutes,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch,▁▂▄▅▇█
epoch_loss,█▇▆▄▃▁
epoch_time,▇██▇█▁
eval_loss,▂▂▂▁▁▁▁▁▁▁▁▁▁▁▃▃▃▂▅▅▅▅▅█
eval_time,▁▁▂▁▁▁▁▇██▇▂▁▁▁▁▂▁▂▂▁▂▂▁
gpu_memory_percent,▁███████████████████████████████████████
+10,...


## Optimize GPU consumption to max of its capacity

In [6]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import wandb
import time
import psutil
import GPUtil
from datetime import datetime

# Configuration
max_steps = 500
batch_size = 8
learning_rate = 5e-5
logging_steps = 100
save_steps = 400
eval_steps = 200
n_epochs = 5
max_length = 756
run_name = f'llmops-{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}'
# Initialize wandb with more config
wandb.init(
    project="llmops-test",
    name=run_name,
    config={
        "max_steps": max_steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_epochs": n_epochs,
        "max_length": max_length
    }
)

# Prepare data
train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(ds["test"], batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()
tokenizer.pad_token = tokenizer.eos_token
model.train()

# Timing and monitoring setup
start_time = time.time()
step = 0
total_batches = len(train_loader)

print(f"Starting training: {n_epochs} epochs, {total_batches} batches per epoch")
print(f"Total steps planned: {min(max_steps, n_epochs * total_batches)}")

for epoch in range(n_epochs):
    if step >= max_steps:
        break
        
    epoch_start_time = time.time()
    epoch_loss = 0
    epoch_batches = 0
    
    print(f"\n=== Epoch {epoch + 1}/{n_epochs} ===")
    
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            break
            
        batch_start_time = time.time()
        
        inputs = tokenizer(
            batch["question"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )
        labels = tokenizer(
            batch["answer"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )["input_ids"]
        
        # Replace padding tokens in labels with -100
        labels[labels == tokenizer.pad_token_id] = -100
        
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_time = time.time() - batch_start_time
        epoch_loss += loss.item()
        epoch_batches += 1
        
        # Comprehensive logging
        if step % logging_steps == 0:
            elapsed_time = time.time() - start_time
            
            # GPU monitoring
            gpu_stats = {}
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats = {
                        "gpu_utilization": gpu.load * 100,
                        "gpu_memory_used": gpu.memoryUsed,
                        "gpu_memory_total": gpu.memoryTotal,
                        "gpu_memory_percent": (gpu.memoryUsed / gpu.memoryTotal) * 100,
                        "gpu_temperature": gpu.temperature
                    }
            except:
                pass
            
            # System stats
            cpu_percent = psutil.cpu_percent()
            memory = psutil.virtual_memory()
            
            # Training progress
            progress = (step / max_steps) * 100
            estimated_total_time = (elapsed_time / max(step, 1)) * max_steps
            eta = estimated_total_time - elapsed_time
            
            log_dict = {
                "train_loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]['lr'],
                "elapsed_time_minutes": elapsed_time / 60,
                "cpu_percent": cpu_percent,
                "memory_percent": memory.percent,
                "memory_used_gb": memory.used / (1024**3),
                **gpu_stats
            }
            
            wandb.log(log_dict)
            
            print(f"Step {step}/{max_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Batch time: {batch_time:.2f}s | "
                  f"ETA: {eta/60:.1f}min | "
                  f"GPU: {gpu_stats.get('gpu_memory_percent', 0):.1f}%")

        # Evaluation
        if step % eval_steps == 0 and step > 0:
            print(f"Running evaluation at step {step}...")
            eval_start_time = time.time()
            
            model.eval()
            eval_loss = 0
            eval_batches = 0
            with torch.no_grad():
                for eval_batch in eval_loader:
                    eval_inputs = tokenizer(
                        eval_batch["question"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )
                    eval_labels = tokenizer(
                        eval_batch["answer"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )["input_ids"]
                    
                    eval_labels[eval_labels == tokenizer.pad_token_id] = -100
                    
                    eval_inputs = {k: v.to(model.device) for k, v in eval_inputs.items()}
                    eval_labels = eval_labels.to(model.device)
                    
                    with autocast(device_type='cuda', dtype=torch.float16):
                        eval_outputs = model(**eval_inputs, labels=eval_labels)
                        eval_loss += eval_outputs.loss.item()
                        eval_batches += 1
            
            avg_eval_loss = eval_loss / eval_batches
            eval_time = time.time() - eval_start_time
            
            wandb.log({
                "eval_loss": avg_eval_loss,
                "eval_time": eval_time,
                "step": step
            })
            
            print(f"Evaluation complete: Loss {avg_eval_loss:.4f} (took {eval_time:.2f}s)")
            model.train()

        # Save checkpoint
        if step % save_steps == 0 and step > 0:
            checkpoint_path = f"{output_dir}/model_step_{step}.pt"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint: {checkpoint_path}")
            wandb.log({"checkpoint_saved": step})

        step += 1

    # End of epoch logging
    epoch_time = time.time() - epoch_start_time
    avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
    
    wandb.log({
        "epoch_loss": avg_epoch_loss,
        "epoch_time": epoch_time,
        "epoch": epoch + 1,
        "batches_per_epoch": epoch_batches
    })
    
    print(f"Epoch {epoch + 1} complete: Avg loss {avg_epoch_loss:.4f}, Time: {epoch_time:.2f}s")

# Training complete
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
wandb.log({"total_training_time_minutes": total_time/60})

wandb.finish()

wandb: Currently logged in as: creigner (creigner-axa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training: 5 epochs, 935 batches per epoch
Total steps planned: 500

=== Epoch 1/5 ===
Step 0/500 (0.0%) | Loss: 10.6978 | Batch time: 1.00s | ETA: 8.3min | GPU: 77.9%
Step 100/500 (20.0%) | Loss: 4.4882 | Batch time: 0.65s | ETA: 4.8min | GPU: 97.6%
Step 200/500 (40.0%) | Loss: 4.1246 | Batch time: 0.65s | ETA: 3.6min | GPU: 97.6%
Running evaluation at step 200...
Evaluation complete: Loss 4.4289 (took 34.73s)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.11 GiB. GPU 0 has a total capacity of 21.98 GiB of which 30.44 MiB is free. Process 8930 has 21.94 GiB memory in use. Of the allocated memory 18.94 GiB is allocated by PyTorch, and 2.69 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

- GPU setup = base ; batch size = 2 ; GPU memory consumption 36% ; time to train = 5 mns
- GPU setup = base ; batch size = 8 ; GPU memory consumption 78% ; time to train = 8mns 
- GPU setup = optimized ; batch size = 8 ; GPU memory consumption 78% ; time to train = 8mns 

## Reduce GPU memory consumption

In [7]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import wandb
import time
import psutil
import GPUtil
from datetime import datetime

model.config.use_cache = False
model.gradient_checkpointing_enable()
# Configuration
max_steps = 1000
batch_size = 8
learning_rate = 5e-5
logging_steps = 100
save_steps = 400
eval_steps = 200
n_epochs = 10
max_length = 512
run_name = f'llmops-{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}'
# Initialize wandb with more config
wandb.init(
    project="llmops-test",
    name=run_name,
    config={
        "max_steps": max_steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_epochs": n_epochs,
        "max_length": max_length
    }
)

# Prepare data
train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(ds["test"], batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()
tokenizer.pad_token = tokenizer.eos_token
model.train()

# Timing and monitoring setup
start_time = time.time()
step = 0
total_batches = len(train_loader)

print(f"Starting training: {n_epochs} epochs, {total_batches} batches per epoch")
print(f"Total steps planned: {min(max_steps, n_epochs * total_batches)}")

for epoch in range(n_epochs):
    if step >= max_steps:
        break
        
    epoch_start_time = time.time()
    epoch_loss = 0
    epoch_batches = 0
    
    print(f"\n=== Epoch {epoch + 1}/{n_epochs} ===")
    
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            break
            
        batch_start_time = time.time()
        
        inputs = tokenizer(
            batch["question"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )
        labels = tokenizer(
            batch["answer"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )["input_ids"]
        
        # Replace padding tokens in labels with -100
        labels[labels == tokenizer.pad_token_id] = -100
        
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_time = time.time() - batch_start_time
        epoch_loss += loss.item()
        epoch_batches += 1
        
        # Comprehensive logging
        if step % logging_steps == 0:
            elapsed_time = time.time() - start_time
            
            # GPU monitoring
            gpu_stats = {}
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats = {
                        "gpu_utilization": gpu.load * 100,
                        "gpu_memory_used": gpu.memoryUsed,
                        "gpu_memory_total": gpu.memoryTotal,
                        "gpu_memory_percent": (gpu.memoryUsed / gpu.memoryTotal) * 100,
                        "gpu_temperature": gpu.temperature
                    }
            except:
                pass
            
            # System stats
            cpu_percent = psutil.cpu_percent()
            memory = psutil.virtual_memory()
            
            # Training progress
            progress = (step / max_steps) * 100
            estimated_total_time = (elapsed_time / max(step, 1)) * max_steps
            eta = estimated_total_time - elapsed_time
            
            log_dict = {
                "train_loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]['lr'],
                "elapsed_time_minutes": elapsed_time / 60,
                "cpu_percent": cpu_percent,
                "memory_percent": memory.percent,
                "memory_used_gb": memory.used / (1024**3),
                **gpu_stats
            }
            
            wandb.log(log_dict)
            
            print(f"Step {step}/{max_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Batch time: {batch_time:.2f}s | "
                  f"ETA: {eta/60:.1f}min | "
                  f"GPU: {gpu_stats.get('gpu_memory_percent', 0):.1f}%")

        # Evaluation
        if step % eval_steps == 0 and step > 0:
            print(f"Running evaluation at step {step}...")
            eval_start_time = time.time()
            
            model.eval()
            eval_loss = 0
            eval_batches = 0
            with torch.no_grad():
                for eval_batch in eval_loader:
                    eval_inputs = tokenizer(
                        eval_batch["question"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )
                    eval_labels = tokenizer(
                        eval_batch["answer"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )["input_ids"]
                    
                    eval_labels[eval_labels == tokenizer.pad_token_id] = -100
                    
                    eval_inputs = {k: v.to(model.device) for k, v in eval_inputs.items()}
                    eval_labels = eval_labels.to(model.device)
                    
                    with autocast(device_type='cuda'):
                        eval_outputs = model(**eval_inputs, labels=eval_labels)
                        eval_loss += eval_outputs.loss.item()
                        eval_batches += 1
            
            avg_eval_loss = eval_loss / eval_batches
            eval_time = time.time() - eval_start_time
            
            wandb.log({
                "eval_loss": avg_eval_loss,
                "eval_time": eval_time,
                "step": step
            })
            
            print(f"Evaluation complete: Loss {avg_eval_loss:.4f} (took {eval_time:.2f}s)")
            model.train()

        # Save checkpoint
        if step % save_steps == 0 and step > 0:
            checkpoint_path = f"{output_dir}/model_step_{step}.pt"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint: {checkpoint_path}")
            wandb.log({"checkpoint_saved": step})

        step += 1

    # End of epoch logging
    epoch_time = time.time() - epoch_start_time
    avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
    
    wandb.log({
        "epoch_loss": avg_epoch_loss,
        "epoch_time": epoch_time,
        "epoch": epoch + 1,
        "batches_per_epoch": epoch_batches
    })
    
    print(f"Epoch {epoch + 1} complete: Avg loss {avg_epoch_loss:.4f}, Time: {epoch_time:.2f}s")

# Training complete
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
wandb.log({"total_training_time_minutes": total_time/60})

wandb.finish()

wandb: Currently logged in as: creigner (creigner-axa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training: 10 epochs, 935 batches per epoch
Total steps planned: 1000

=== Epoch 1/10 ===
Step 0/1000 (0.0%) | Loss: 10.3504 | Batch time: 0.90s | ETA: 15.0min | GPU: 25.0%
Step 100/1000 (10.0%) | Loss: 4.4550 | Batch time: 0.53s | ETA: 9.0min | GPU: 45.3%
Step 200/1000 (20.0%) | Loss: 4.3273 | Batch time: 0.53s | ETA: 7.9min | GPU: 45.3%
Running evaluation at step 200...
Evaluation complete: Loss 4.4131 (took 22.50s)
Step 300/1000 (30.0%) | Loss: 4.2477 | Batch time: 0.53s | ETA: 7.8min | GPU: 48.6%
Step 400/1000 (40.0%) | Loss: 4.4207 | Batch time: 0.53s | ETA: 6.5min | GPU: 48.6%
Running evaluation at step 400...
Evaluation complete: Loss 4.3373 (took 22.52s)
Saved checkpoint: ./test_gmsk8_smol/model_step_400.pt
Step 500/1000 (50.0%) | Loss: 4.3258 | Batch time: 0.53s | ETA: 6.0min | GPU: 48.6%
Step 600/1000 (60.0%) | Loss: 4.4965 | Batch time: 0.53s | ETA: 4.6min | GPU: 48.6%
Running evaluation at step 600...
Evaluation complete: Loss 4.2899 (took 22.51s)
Step 700/1000 (70.

batches_per_epoch,█▁
checkpoint_saved,▁█
cpu_percent,▃▁███▇███▆
elapsed_time_minutes,▁▂▂▃▄▅▅▆▇█
epoch,▁█
epoch_loss,█▁
epoch_time,█▁
eval_loss,█▄▂▁
eval_time,▁▇▂█
gpu_memory_percent,▁▇▇███████
+10,...


In [6]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import wandb
import time
import psutil
import GPUtil
from datetime import datetime

model.config.use_cache = False
model.gradient_checkpointing_enable()
# Configuration
max_steps = 1000
batch_size = 16
learning_rate = 5e-5
logging_steps = 100
save_steps = 400
eval_steps = 200
n_epochs = 10
max_length = 512
run_name = f'llmops-{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}'
# Initialize wandb with more config
wandb.init(
    project="llmops-test",
    name=run_name,
    config={
        "max_steps": max_steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_epochs": n_epochs,
        "max_length": max_length
    }
)

# Prepare data
train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(ds["test"], batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()
tokenizer.pad_token = tokenizer.eos_token
model.train()

# Timing and monitoring setup
start_time = time.time()
step = 0
total_batches = len(train_loader)

print(f"Starting training: {n_epochs} epochs, {total_batches} batches per epoch")
print(f"Total steps planned: {min(max_steps, n_epochs * total_batches)}")

for epoch in range(n_epochs):
    if step >= max_steps:
        break
        
    epoch_start_time = time.time()
    epoch_loss = 0
    epoch_batches = 0
    
    print(f"\n=== Epoch {epoch + 1}/{n_epochs} ===")
    
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            break
            
        batch_start_time = time.time()
        
        inputs = tokenizer(
            batch["question"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )
        labels = tokenizer(
            batch["answer"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )["input_ids"]
        
        # Replace padding tokens in labels with -100
        labels[labels == tokenizer.pad_token_id] = -100
        
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_time = time.time() - batch_start_time
        epoch_loss += loss.item()
        epoch_batches += 1
        
        # Comprehensive logging
        if step % logging_steps == 0:
            elapsed_time = time.time() - start_time
            
            # GPU monitoring
            gpu_stats = {}
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats = {
                        "gpu_utilization": gpu.load * 100,
                        "gpu_memory_used": gpu.memoryUsed,
                        "gpu_memory_total": gpu.memoryTotal,
                        "gpu_memory_percent": (gpu.memoryUsed / gpu.memoryTotal) * 100,
                        "gpu_temperature": gpu.temperature
                    }
            except:
                pass
            
            # System stats
            cpu_percent = psutil.cpu_percent()
            memory = psutil.virtual_memory()
            
            # Training progress
            progress = (step / max_steps) * 100
            estimated_total_time = (elapsed_time / max(step, 1)) * max_steps
            eta = estimated_total_time - elapsed_time
            
            log_dict = {
                "train_loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]['lr'],
                "elapsed_time_minutes": elapsed_time / 60,
                "cpu_percent": cpu_percent,
                "memory_percent": memory.percent,
                "memory_used_gb": memory.used / (1024**3),
                **gpu_stats
            }
            
            wandb.log(log_dict)
            
            print(f"Step {step}/{max_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Batch time: {batch_time:.2f}s | "
                  f"ETA: {eta/60:.1f}min | "
                  f"GPU: {gpu_stats.get('gpu_memory_percent', 0):.1f}%")

        # Evaluation
        if step % eval_steps == 0 and step > 0:
            print(f"Running evaluation at step {step}...")
            eval_start_time = time.time()
            
            model.eval()
            eval_loss = 0
            eval_batches = 0
            with torch.no_grad():
                for eval_batch in eval_loader:
                    eval_inputs = tokenizer(
                        eval_batch["question"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )
                    eval_labels = tokenizer(
                        eval_batch["answer"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )["input_ids"]
                    
                    eval_labels[eval_labels == tokenizer.pad_token_id] = -100
                    
                    eval_inputs = {k: v.to(model.device) for k, v in eval_inputs.items()}
                    eval_labels = eval_labels.to(model.device)
                    
                    with autocast(device_type='cuda'):
                        eval_outputs = model(**eval_inputs, labels=eval_labels)
                        eval_loss += eval_outputs.loss.item()
                        eval_batches += 1
            
            avg_eval_loss = eval_loss / eval_batches
            eval_time = time.time() - eval_start_time
            
            wandb.log({
                "eval_loss": avg_eval_loss,
                "eval_time": eval_time,
                "step": step
            })
            
            print(f"Evaluation complete: Loss {avg_eval_loss:.4f} (took {eval_time:.2f}s)")
            model.train()

        # Save checkpoint
        if step % save_steps == 0 and step > 0:
            checkpoint_path = f"{output_dir}/model_step_{step}.pt"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint: {checkpoint_path}")
            wandb.log({"checkpoint_saved": step})

        step += 1

    # End of epoch logging
    epoch_time = time.time() - epoch_start_time
    avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
    
    wandb.log({
        "epoch_loss": avg_epoch_loss,
        "epoch_time": epoch_time,
        "epoch": epoch + 1,
        "batches_per_epoch": epoch_batches
    })
    
    print(f"Epoch {epoch + 1} complete: Avg loss {avg_epoch_loss:.4f}, Time: {epoch_time:.2f}s")

# Training complete
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
wandb.log({"total_training_time_minutes": total_time/60})

wandb.finish()

wandb: Currently logged in as: creigner (creigner-axa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training: 10 epochs, 468 batches per epoch
Total steps planned: 1000

=== Epoch 1/10 ===
Step 0/1000 (0.0%) | Loss: 11.1031 | Batch time: 1.32s | ETA: 22.0min | GPU: 39.1%
Step 100/1000 (10.0%) | Loss: 4.3783 | Batch time: 1.02s | ETA: 16.4min | GPU: 59.1%
Step 200/1000 (20.0%) | Loss: 4.2374 | Batch time: 1.02s | ETA: 14.5min | GPU: 59.1%
Running evaluation at step 200...
Evaluation complete: Loss 4.3697 (took 21.90s)
Step 300/1000 (30.0%) | Loss: 4.3476 | Batch time: 1.02s | ETA: 13.5min | GPU: 59.1%
Step 400/1000 (40.0%) | Loss: 4.4626 | Batch time: 1.02s | ETA: 11.4min | GPU: 59.1%
Running evaluation at step 400...
Evaluation complete: Loss 4.2749 (took 21.90s)
Saved checkpoint: ./test_gmsk8_smol/model_step_400.pt
Epoch 1 complete: Avg loss 4.5374, Time: 565.77s

=== Epoch 2/10 ===
Step 500/1000 (50.0%) | Loss: 4.1496 | Batch time: 1.02s | ETA: 10.0min | GPU: 59.2%
Step 600/1000 (60.0%) | Loss: 4.1546 | Batch time: 1.02s | ETA: 7.9min | GPU: 59.2%
Running evaluation at ste

batches_per_epoch,██▁
checkpoint_saved,▁█
cpu_percent,▄████▂▁▇▅▁
elapsed_time_minutes,▁▂▂▃▄▅▆▆▇█
epoch,▁▅█
epoch_loss,█▃▁
epoch_time,██▁
eval_loss,█▃▂▁
eval_time,██▃▁
gpu_memory_percent,▁█████████
+10,...


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import wandb
import time
import psutil
import GPUtil
from datetime import datetime

model.config.use_cache = False
model.gradient_checkpointing_enable()
# Configuration
max_steps = 1000
batch_size = 16 + 8 * 2
learning_rate = 5e-5
logging_steps = 100
save_steps = 400
eval_steps = 200
n_epochs = 10
max_length = 512
run_name = f'llmops-{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}'
# Initialize wandb with more config
wandb.init(
    project="llmops-test",
    name=run_name,
    config={
        "max_steps": max_steps,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "n_epochs": n_epochs,
        "max_length": max_length
    }
)

# Prepare data
train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(ds["test"], batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()
tokenizer.pad_token = tokenizer.eos_token
model.train()

# Timing and monitoring setup
start_time = time.time()
step = 0
total_batches = len(train_loader)

print(f"Starting training: {n_epochs} epochs, {total_batches} batches per epoch")
print(f"Total steps planned: {min(max_steps, n_epochs * total_batches)}")

for epoch in range(n_epochs):
    if step >= max_steps:
        break
        
    epoch_start_time = time.time()
    epoch_loss = 0
    epoch_batches = 0
    
    print(f"\n=== Epoch {epoch + 1}/{n_epochs} ===")
    
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            break
            
        batch_start_time = time.time()
        
        inputs = tokenizer(
            batch["question"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )
        labels = tokenizer(
            batch["answer"], 
            return_tensors="pt", 
            padding="max_length",
            truncation=True, 
            max_length=max_length
        )["input_ids"]
        
        # Replace padding tokens in labels with -100
        labels[labels == tokenizer.pad_token_id] = -100
        
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_time = time.time() - batch_start_time
        epoch_loss += loss.item()
        epoch_batches += 1
        
        # Comprehensive logging
        if step % logging_steps == 0:
            elapsed_time = time.time() - start_time
            
            # GPU monitoring
            gpu_stats = {}
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats = {
                        "gpu_utilization": gpu.load * 100,
                        "gpu_memory_used": gpu.memoryUsed,
                        "gpu_memory_total": gpu.memoryTotal,
                        "gpu_memory_percent": (gpu.memoryUsed / gpu.memoryTotal) * 100,
                        "gpu_temperature": gpu.temperature
                    }
            except:
                pass
            
            # System stats
            cpu_percent = psutil.cpu_percent()
            memory = psutil.virtual_memory()
            
            # Training progress
            progress = (step / max_steps) * 100
            estimated_total_time = (elapsed_time / max(step, 1)) * max_steps
            eta = estimated_total_time - elapsed_time
            
            log_dict = {
                "train_loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]['lr'],
                "elapsed_time_minutes": elapsed_time / 60,
                "cpu_percent": cpu_percent,
                "memory_percent": memory.percent,
                "memory_used_gb": memory.used / (1024**3),
                **gpu_stats
            }
            
            wandb.log(log_dict)
            
            print(f"Step {step}/{max_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Batch time: {batch_time:.2f}s | "
                  f"ETA: {eta/60:.1f}min | "
                  f"GPU: {gpu_stats.get('gpu_memory_percent', 0):.1f}%")

        # Evaluation
        if step % eval_steps == 0 and step > 0:
            print(f"Running evaluation at step {step}...")
            eval_start_time = time.time()
            
            model.eval()
            eval_loss = 0
            eval_batches = 0
            with torch.no_grad():
                for eval_batch in eval_loader:
                    eval_inputs = tokenizer(
                        eval_batch["question"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )
                    eval_labels = tokenizer(
                        eval_batch["answer"], 
                        return_tensors="pt", 
                        padding="max_length",
                        truncation=True, 
                        max_length=max_length
                    )["input_ids"]
                    
                    eval_labels[eval_labels == tokenizer.pad_token_id] = -100
                    
                    eval_inputs = {k: v.to(model.device) for k, v in eval_inputs.items()}
                    eval_labels = eval_labels.to(model.device)
                    
                    with autocast(device_type='cuda'):
                        eval_outputs = model(**eval_inputs, labels=eval_labels)
                        eval_loss += eval_outputs.loss.item()
                        eval_batches += 1
            
            avg_eval_loss = eval_loss / eval_batches
            eval_time = time.time() - eval_start_time
            
            wandb.log({
                "eval_loss": avg_eval_loss,
                "eval_time": eval_time,
                "step": step
            })
            
            print(f"Evaluation complete: Loss {avg_eval_loss:.4f} (took {eval_time:.2f}s)")
            model.train()

        # Save checkpoint
        if step % save_steps == 0 and step > 0:
            checkpoint_path = f"{output_dir}/model_step_{step}.pt"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint: {checkpoint_path}")
            wandb.log({"checkpoint_saved": step})

        step += 1

    # End of epoch logging
    epoch_time = time.time() - epoch_start_time
    avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
    
    wandb.log({
        "epoch_loss": avg_epoch_loss,
        "epoch_time": epoch_time,
        "epoch": epoch + 1,
        "batches_per_epoch": epoch_batches
    })
    
    print(f"Epoch {epoch + 1} complete: Avg loss {avg_epoch_loss:.4f}, Time: {epoch_time:.2f}s")

# Training complete
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
wandb.log({"total_training_time_minutes": total_time/60})

wandb.finish()

wandb: Currently logged in as: creigner (creigner-axa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training: 10 epochs, 234 batches per epoch
Total steps planned: 1000

=== Epoch 1/10 ===
Step 0/1000 (0.0%) | Loss: 11.0205 | Batch time: 2.28s | ETA: 38.1min | GPU: 67.9%
Step 100/1000 (10.0%) | Loss: 4.5741 | Batch time: 1.94s | ETA: 30.3min | GPU: 94.6%
Step 200/1000 (20.0%) | Loss: 4.3239 | Batch time: 1.94s | ETA: 26.8min | GPU: 94.6%
Running evaluation at step 200...
Evaluation complete: Loss 4.3260 (took 20.02s)
Epoch 1 complete: Avg loss 4.7312, Time: 487.22s

=== Epoch 2/10 ===
Step 300/1000 (30.0%) | Loss: 4.3270 | Batch time: 1.94s | ETA: 24.2min | GPU: 94.6%
Step 400/1000 (40.0%) | Loss: 4.0966 | Batch time: 1.94s | ETA: 20.5min | GPU: 94.6%
Running evaluation at step 400...
Evaluation complete: Loss 4.2564 (took 20.02s)
Saved checkpoint: ./test_gmsk8_smol/model_step_400.pt
Epoch 2 complete: Avg loss 4.2093, Time: 504.45s

=== Epoch 3/10 ===
Step 500/1000 (50.0%) | Loss: 4.0427 | Batch time: 1.94s | ETA: 17.6min | GPU: 94.6%
Step 600/1000 (60.0%) | Loss: 4.1961 | B

batches_per_epoch,████▁
checkpoint_saved,▁█
cpu_percent,▁▅▅▅▅▄▅█▅▄
elapsed_time_minutes,▁▂▂▃▄▅▆▆▇█
epoch,▁▃▅▆█
epoch_loss,█▄▃▂▁
epoch_time,████▁
eval_loss,█▃▁▁
eval_time,▁▃▂█
gpu_memory_percent,▁█████████
+10,...


With max steps = 1000, batch_size=8
- base model running in 8mns with up to 76% GPU (17GB) consumption
- base model running in 8 mns with up to 98% GPU (22GB) consumption with optimized consumption
- GPU optimized running in 12 mns with up to 45% (12GB) consumption with gradient checkpointers and caching disabled 
- With batch size= 16 up to 60% 14GB in 18mns
- With batch size= 32 up to 98% 22GB in 32mns ?

So why should I increase my training time as well as batch size ?
Well because although your training loss remains similar, your validation loss is much better (and closer to training).
See in the below chart that the green line (small batch) shows an eval loss higher than the training loss while the purple chart shows a better eval loss that is reached much faster.

